### Getting base Phase A dataframe
* Alligning raw LLaMA output with annotations to get clip name labels and audio start/end
* Alligning action scores (quality score 2) with indices

In [3]:
import numpy as np
import pandas as pd
import ast

stage2 = pd.read_csv("results/stage2_llama3_assistant.csv")
anno = pd.read_csv("results/tvad_anno_context-9-24_face-0.2-0.4_scale_thread.csv")
actn_sc = pd.read_csv("results/action_scores.csv")

anno_lookup = (anno[["anno_idx", "tvad_name", "AD_start", "AD_end"]]
               .drop_duplicates(subset="anno_idx")
               .set_index("anno_idx"))

CAND_LABELS = ["A", "B", "C", "D", "E"]

rows = []

for _, row in stage2.iterrows():
    candidates = ast.literal_eval(row["text_gen"])
    scene_dur = row["end"] - row["start"]

    idx = row["anno_idx"]

    if idx in anno_lookup.index:
        tvad_name = anno_lookup.loc[idx, "tvad_name"]
        AD_start = anno_lookup.loc[idx, "AD_start"]
        AD_end = anno_lookup.loc[idx, "AD_end"]
    else:
        tvad_name = None
        AD_start = None
        AD_end = None

    for i, cand_text in enumerate(candidates):
        n_tokens = len(str(cand_text).split())
        TTS_speech_dur = n_tokens / 2.5

        rows.append({
            "anno_idx": idx,
            "scene_id": f"s{int(idx)}",
            "cand_id": CAND_LABELS[i],
            "scene_dur": scene_dur,
            "n_tokens": n_tokens,
            "TTS_speech_dur": TTS_speech_dur,
            "human_AD_text": row["text_gt"],
            "gen_AD_text": cand_text,
            "tvad_name": tvad_name,
            "AD_start": AD_start,
            "AD_end": AD_end,
        })

phaseA = pd.DataFrame(rows)

In [4]:
phaseA = phaseA.merge(actn_sc[["anno_idx", "text_gen", "action_score"]],
    left_on=["anno_idx", "gen_AD_text"],
    right_on=["anno_idx", "text_gen"],
    how="left"
).drop(columns=["text_gen"]).rename(columns={"action_score": "quality_score"})

### Extracting more features using sentence embeddings
* Getting quality score 1 (human_AD_sim) and consensus_score using sentence transformers
* Getting NLP features based on candidate ADs only
* Renaming variables for clarity and saving

In [6]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")

# quality score: similarity between human & AI AD text
human_embs = model.encode(phaseA["human_AD_text"].tolist(), batch_size=64, show_progress_bar=True)
gen_embs = model.encode(phaseA["gen_AD_text"].tolist(), batch_size=64, show_progress_bar=True)
phaseA["human_AD_sim"] = util.cos_sim(gen_embs, human_embs).diagonal().tolist()

# consensus score: similarity between candidate ADs within each scene
all_texts = phaseA["gen_AD_text"].tolist()
all_embs = model.encode(all_texts, batch_size=64, show_progress_bar=True)

scores = np.zeros(len(phaseA))
for _, group in phaseA.groupby("scene_id"):
    idx = group.index
    embs = all_embs[idx]
    sim = util.cos_sim(embs, embs).numpy()
    np.fill_diagonal(sim, 0)
    scores[idx] = sim.mean(axis=1)

phaseA["consensus_score"] = scores

/home/mellie/miniconda3/envs/AD_selection/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 238/238 [02:34<00:00,  1.54it/s]


In [7]:
import spacy

nlp = spacy.load("en_core_web_sm")

ACTION_VERBS = {  # from the shot by shot repo
    'look', 'walk', 'turn', 'stare', 'take', 'hold', 'smile', 'leave',
    'pull', 'watch', 'open', 'go', 'step', 'get', 'enter'
} 

# parse all candidate texts at once
texts = phaseA["gen_AD_text"].tolist()
docs = list(nlp.pipe(texts, batch_size=64))

n_action_verbs = []
verb_density = []
noun_ratio = []
pronoun_to_noun_ratio = []
type_token_ratio = []

for doc, n_tok in zip(docs, phaseA["n_tokens"]):
    tokens = [t for t in doc if not t.is_space]
    total = max(len(tokens), 1)
    
    # action verb count, lemmatized
    av_count = sum(1 for t in tokens if t.lemma_.lower() in ACTION_VERBS)
    n_action_verbs.append(av_count)
    verb_density.append(av_count / total)
    
    # nouns and pronouns
    nouns = [t for t in tokens if t.pos_ == "NOUN"]
    pronouns = [t for t in tokens if t.pos_ == "PRON"]
    noun_count = max(len(nouns), 1)
    noun_ratio.append(len(nouns) / total)
    pronoun_to_noun_ratio.append(len(pronouns) / noun_count)
    
    # type-token ratio (unique / total words)
    words = [t.text.lower() for t in tokens if t.is_alpha]
    ttr = len(set(words)) / max(len(words), 1)
    type_token_ratio.append(ttr)

phaseA["n_action_verbs"] = n_action_verbs
phaseA["verb_density"] = verb_density
phaseA["noun_ratio"] = noun_ratio
phaseA["pronoun_to_noun_ratio"] = pronoun_to_noun_ratio
phaseA["type_token_ratio"] = type_token_ratio

# length difference from scene mean
scene_mean_tokens = phaseA.groupby("scene_id")["n_tokens"].transform("mean")
phaseA["len_diff_from_mean"] = phaseA["n_tokens"] - scene_mean_tokens

In [8]:
phaseA.rename(columns={"human_AD_sim": "quality_score1", "quality_score": "quality_score2"}, inplace=True)
phaseA.to_csv("phaseA.csv", index=False)

In [9]:
phaseA.head(2)

,anno_idx,scene_id,cand_id,scene_dur,n_tokens,TTS_speech_dur,human_AD_text,gen_AD_text,tvad_name,AD_start,AD_end,quality_score2,quality_score1,consensus_score,n_action_verbs,verb_density,noun_ratio,pronoun_to_noun_ratio,type_token_ratio,len_diff_from_mean
0,0,s0,A,1.862,7,2.8,His umbrella springs open between them.,"Phoebe holds Rachel's hand, looking at her.",friends_s01e01_seg02_clip_04,240.25,242.112,0.336894,0.172184,0.609855,2,0.200000,0.100000,1.0,1.0,-1.2
1,0,s0,B,1.862,8,3.2,His umbrella springs open between them.,"Rachel kneels, holding a cup, as Phoebe gazes.",friends_s01e01_seg02_clip_04,240.25,242.112,0.349908,0.164244,0.583783,1,0.090909,0.181818,0.0,1.0,-0.2


### Training selector model classificaiton on Q1 (human_AD_sim)
* First get baselines, random chance being the important one
* Then do logreg, rf, and xgboost

In [59]:
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv("results/phaseA.csv")

# create a binary target var based on response var
df["best_cand"] = 0
df.loc[df.groupby("scene_id")["quality_score1"].idxmax(), "best_cand"] = 1

features = ["n_tokens", "TTS_speech_dur", "consensus_score",
            "n_action_verbs", "verb_density", "noun_ratio",
            "pronoun_to_noun_ratio", "type_token_ratio", "len_diff_from_mean"]

X = df[features].values
y = df["best_cand"].values

# making sure i get exactly 5 candidate groups per scene
unique_scenes = df["scene_id"].unique()
scene_to_int = {s: i for i, s in enumerate(unique_scenes)}
groups = df["scene_id"].map(scene_to_int).values

gkf = GroupKFold(n_splits=5)

# scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# baselines
print("Baselines")
baselines = {}

for s in ["shortest", "random", "first"]:
    correct = 0
    for _, group in df.groupby("scene_id"):
        true_best = group["quality_score1"].idxmax()
        if s == "shortest":
            picked = group["n_tokens"].idxmin()
        elif s == "random":
            picked = group.sample(1).index[0]
        elif s == "first":
            picked = group.index[0]
        correct += int(picked == true_best)
    baselines[s] = correct / df["scene_id"].nunique()
    print(s, baselines[s])

# CV object to make sure all models select by scene (5 rows), not by row
def selector_accuracy_cv(model, X, df, groups):
    scene_correct = []
    for train_idx, test_idx in GroupKFold(n_splits=5).split(X, y, groups):
        model.fit(X[train_idx], y[train_idx])
        test_df = df.iloc[test_idx].copy()
        test_df["pred_prob"] = model.predict_proba(X[test_idx])[:, 1]
        pred_best = test_df.groupby("scene_id")["pred_prob"].idxmax()
        true_best = test_df.groupby("scene_id")["quality_score1"].idxmax()
        scene_correct.append((pred_best == true_best).mean())
    return np.mean(scene_correct), np.std(scene_correct)

Baselines
shortest 0.21890714046262152
random 0.2011397921555481
first 0.23198122695273216


In [17]:
# logreg L2 with tuned C
logreg_results = {}
for C in [0.001, 0.01, 0.1, 1, 10, 100]:
    m = LogisticRegression(C=C, max_iter=2000, random_state=12)
    acc, std = selector_accuracy_cv(m, X_scaled, df, groups)
    logreg_results[C] = acc

best_C = max(logreg_results, key=logreg_results.get)
tuned_logreg = LogisticRegression(C=best_C, max_iter=2000, random_state=12)
logreg_acc, logreg_std = selector_accuracy_cv(tuned_logreg, X_scaled, df, groups)
print("best C:", best_C)
print("acc:", logreg_acc, "±", logreg_std)

best C: 10
acc: 0.2504069564826369 ± 0.01598124442656204


In [20]:
# rf tuning
rf_grid = {
    "n_estimators": [100, 200],
    "max_samples": [0.6, 0.75, 0.9],
    "max_features": [0.3, 0.5, 0.7],
    "max_depth": [10, 20, None],
    "min_samples_leaf": [1, 3, 5]
}

rf_base = RandomForestClassifier(random_state=12)
rscv_rf = RandomizedSearchCV(rf_base, rf_grid, n_iter=5, cv=5,
                              scoring="accuracy", random_state=12)
rscv_rf.fit(X, y)
print("best params:", rscv_rf.best_params_)
tuned_rf = rscv_rf.best_estimator_
rf_acc, rf_std = selector_accuracy_cv(tuned_rf, X, df, groups)
print("acc:", rf_acc, "±", rf_std)

best params: {'n_estimators': 200, 'min_samples_leaf': 5, 'max_samples': 0.6, 'max_features': 0.3, 'max_depth': 10}
acc: 0.23768394545434107 ± 0.006787231393867515


In [24]:
# xgboost tuning
xgb_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 4, 5],
    "subsample": [0.6, 0.75, 0.9],
    "min_child_weight": [1, 3, 5],
    "colsample_bytree": [0.5, 0.7, 1.0]
}
xgb_base = XGBClassifier(random_state=12, eval_metric="logloss")
rscv_xgb = RandomizedSearchCV(xgb_base, xgb_grid, n_iter=15, cv=5,
                              scoring="accuracy", random_state=12)
rscv_xgb.fit(X, y)
print("best XGB params:", rscv_xgb.best_params_)
tuned_xgb = rscv_xgb.best_estimator_
xgb_acc, xgb_std = selector_accuracy_cv(tuned_xgb, X, df, groups)
print("acc:", xgb_acc, "±", xgb_std)

best XGB params: {'subsample': 0.75, 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.01, 'colsample_bytree': 1.0}
acc: 0.2551105640057109 ± 0.003279070022670842


In [26]:
# all printed together
print(baselines)
print("logreg acc:", logreg_acc, "±", logreg_std)
print("rf acc:", rf_acc, "±", rf_std)
print("xgb acc:", xgb_acc, "±", xgb_std)

{'shortest': 0.21890714046262152, 'random': 0.20817968488099228, 'first': 0.23198122695273216}
logreg acc: 0.2504069564826369 ± 0.01598124442656204
rf acc: 0.23768394545434107 ± 0.006787231393867515
xgb acc: 0.2551105640057109 ± 0.003279070022670842


In [28]:
# feature importance for highest accuracy model, xgb
importances = pd.Series(tuned_xgb.feature_importances_, index=features)
importances = importances.sort_values(ascending=False)
print("feature importances for XGB:\n", importances)

feature importances for XGB:
 pronoun_to_noun_ratio    0.156764
consensus_score          0.156104
verb_density             0.155795
noun_ratio               0.145712
len_diff_from_mean       0.135075
type_token_ratio         0.111370
n_tokens                 0.091245
n_action_verbs           0.047935
TTS_speech_dur           0.000000
dtype: float32


### Training selector model classificaiton on Q2 (action score)
* Repeating steps from Q1

In [61]:
# change the target to Q2
df["best_cand"] = 0
df.loc[df.groupby("scene_id")["quality_score2"].idxmax(), "best_cand"] = 1
y = df["best_cand"].values

def selector_accuracy_cv(model, X, df, groups, use_scaled=True):
    X_in = X if use_scaled else X
    scene_correct = []
    for train_idx, test_idx in GroupKFold(n_splits=5).split(X_in, y, groups):
        model.fit(X_in[train_idx], y[train_idx])
        test_df = df.iloc[test_idx].copy()
        test_df["pred_prob"] = model.predict_proba(X_in[test_idx])[:, 1]
        pred_best = test_df.groupby("scene_id")["pred_prob"].idxmax()
        true_best = test_df.groupby("scene_id")["quality_score2"].idxmax()
        scene_correct.append((pred_best == true_best).mean())
    return np.mean(scene_correct), np.std(scene_correct)

In [31]:
# get baselines for Q2
print("Baselines")
baselines = {}

for s in ["shortest", "random", "first"]:
    correct = 0
    for _, group in df.groupby("scene_id"):
        true_best = group["quality_score2"].idxmax()
        if s == "shortest":
            picked = group["n_tokens"].idxmin()
        elif s == "random":
            picked = group.sample(1).index[0]
        elif s == "first":
            picked = group.index[0]
        correct += int(picked == true_best)
    baselines[s] = correct / df["scene_id"].nunique()
    print(s, baselines[s])

Baselines
shortest 0.18035534696614147
random 0.2008045591686222
first 0.22192423734495476


In [ ]:
# almost similar but slightly different

# repeat steps from Q1

In [62]:
# logreg L2 with tuned C
logreg_results = {}
for C in [0.001, 0.01, 0.1, 1, 10, 100]:
    m = LogisticRegression(C=C, max_iter=2000, random_state=12)
    acc, std = selector_accuracy_cv(m, X_scaled, df, groups)
    logreg_results[C] = acc

best_C = max(logreg_results, key=logreg_results.get)
tuned_logreg = LogisticRegression(C=best_C, max_iter=2000, random_state=12)
logreg_acc, logreg_std = selector_accuracy_cv(tuned_logreg, X_scaled, df, groups)
print("best C:", best_C)
print("acc:", logreg_acc, "±", logreg_std)

best C: 0.1
acc: 0.33690263397524534 ± 0.0145799899458466


In [54]:
# rf tuning
rf_grid = {
    "n_estimators": [100, 200],
    "max_samples": [0.6, 0.75, 0.9],
    "max_features": [0.3, 0.5, 0.7],
    "max_depth": [10, 20, None],
    "min_samples_leaf": [1, 3, 5]
}

rf_base = RandomForestClassifier(random_state=12)
rscv_rf = RandomizedSearchCV(rf_base, rf_grid, n_iter=5, cv=5,
                              scoring="accuracy", random_state=12)
rscv_rf.fit(X, y)
print("best params:", rscv_rf.best_params_)
tuned_rf = rscv_rf.best_estimator_
rf_acc, rf_std = selector_accuracy_cv(tuned_rf, X, df, groups)
print("acc:", rf_acc, "±", rf_std)

best params: {'n_estimators': 200, 'min_samples_leaf': 5, 'max_samples': 0.6, 'max_features': 0.3, 'max_depth': 10}
acc: 0.3104223578743831 ± 0.01598082387209106


In [55]:
# xgboost tuning
xgb_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 4, 5],
    "subsample": [0.6, 0.75, 0.9],
    "min_child_weight": [1, 3, 5],
    "colsample_bytree": [0.5, 0.7, 1.0]
}
xgb_base = XGBClassifier(random_state=12, eval_metric="logloss")
rscv_xgb = RandomizedSearchCV(xgb_base, xgb_grid, n_iter=15, cv=5,
                              scoring="accuracy", random_state=12)
rscv_xgb.fit(X, y)
print("best XGB params:", rscv_xgb.best_params_)
tuned_xgb = rscv_xgb.best_estimator_
xgb_acc, xgb_std = selector_accuracy_cv(tuned_xgb, X, df, groups)
print("acc:", xgb_acc, "±", xgb_std)

best XGB params: {'subsample': 0.9, 'n_estimators': 300, 'min_child_weight': 5, 'max_depth': 5, 'learning_rate': 0.01, 'colsample_bytree': 0.7}
acc: 0.3177947905073466 ± 0.015362368989212557


In [56]:
# all printed together
print(baselines)
print("logreg acc:", logreg_acc, "±", logreg_std)
print("rf acc:", rf_acc, "±", rf_std)
print("xgb acc:", xgb_acc, "±", xgb_std)

{'shortest': 0.18035534696614147, 'random': 0.2128729466979551, 'first': 0.22192423734495476}
logreg acc: 0.33690263397524534 ± 0.0145799899458466
rf acc: 0.3104223578743831 ± 0.01598082387209106
xgb acc: 0.3177947905073466 ± 0.015362368989212557


In [58]:
# feature importance for highest accuracy model, logreg
tuned_logreg.fit(X_scaled, y)
importances = pd.Series(np.abs(tuned_logreg.coef_[0]), index=features)
importances = importances.sort_values(ascending=False)
print("feature importances for logreg:\n", importances)

feature importances for logreg:
 n_action_verbs           0.253464
pronoun_to_noun_ratio    0.223718
consensus_score          0.203161
len_diff_from_mean       0.145605
type_token_ratio         0.107632
verb_density             0.043829
noun_ratio               0.011377
n_tokens                 0.009434
TTS_speech_dur           0.009434
dtype: float64


### Training selector model regression on Q1 (human_AD_sim)
* same as classification

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_csv("results/phaseA.csv")
cv = KFold(n_splits=5, shuffle=True, random_state=12)

X = df[features].values
y = df["quality_score1"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

all_preds = {}

# pick candidate with highest predicted score per scene and see if it matches true best
def regression_selector_accuracy(df, pred_col, true_quality_score):
    correct = 0
    total = 0
    temp = df.copy()
    temp["pred"] = pred_col
    for _, group in temp.groupby("scene_id"):
        pred_best = group["pred"].idxmax()
        true_best = group[true_quality_score].idxmax()
        correct += int(pred_best == true_best)
        total += 1
    return correct / total

# CV metrics function
def cv_metrics(model, X, y, cv, label=""):
    preds = cross_val_predict(model, X, y, cv=cv)
    mae = mean_absolute_error(y, preds)
    r2 = r2_score(y, preds)
    return preds, mae, r2

In [68]:
# ridge
alphas = np.logspace(-3, 3, 50)
rcv = RidgeCV(alphas=alphas, cv=cv, scoring="neg_mean_absolute_error")
rcv.fit(X_scaled, y)
preds_ridge, _, _ = cv_metrics(rcv, X_scaled, y, cv, "Ridge")
all_preds["Ridge"] = preds_ridge

In [69]:
# lasso
lcv = LassoCV(alphas=alphas, cv=cv, max_iter=5000)
lcv.fit(X_scaled, y)
preds_lasso, _, _ = cv_metrics(lcv, X_scaled, y, cv, "Lasso")
all_preds["Lasso"] = preds_lasso

In [70]:
# KNN
Ks = np.arange(5, 51, 5)
knn_scores = []
for K in Ks:
    s = np.mean(cross_val_predict(
        KNeighborsRegressor(n_neighbors=K, weights="distance"),
        X_scaled, y, cv=cv
    ))
    knn_scores.append(s)
best_K = Ks[np.argmax(knn_scores)]
knn = KNeighborsRegressor(n_neighbors=best_K, weights="distance")
preds_knn, _, _ = cv_metrics(knn, X_scaled, y, cv, "KNN")
all_preds["KNN"] = preds_knn

In [71]:
# rf
rf_base = RandomForestRegressor(n_estimators=100, random_state=12)
rf_grid = {
    "max_samples": [0.6, 0.75, 0.9],
    "max_features": [0.3, 0.5, 0.7],
    "max_depth": [10, 20, None],
    "min_samples_leaf": [1, 3, 5]
}
rscv_rf = RandomizedSearchCV(rf_base, rf_grid, n_iter=5, cv=cv,
                              scoring="neg_mean_absolute_error",
                              random_state=12)
rscv_rf.fit(X, y)
preds_rf, _, _ = cv_metrics(rscv_rf.best_estimator_, X, y, cv, "RandomForest")
all_preds["RF"] = preds_rf

In [72]:
# xgb
gb_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.05, 0.1, 0.15],
    "max_depth": [3, 4, 5],
    "subsample": [0.75, 0.9],
    "min_samples_leaf": [1, 3],
    "max_features": [0.5, 0.7]
}
rscv_xgb = RandomizedSearchCV(
    XGBRegressor(random_state=12),
    xgb_grid, n_iter=5, cv=cv,
    scoring="neg_mean_absolute_error",
    random_state=12
)
rscv_xgb.fit(X, y)
preds_xgb, _, _ = cv_metrics(rscv_xgb.best_estimator_, X, y, cv, "XGB")
all_preds["GB"] = preds_xgb

In [ ]:
# accuracies printed together
selector_results_q1 = {}

for name, preds in all_preds.items():
    correct = 0
    for _, group in df.groupby("scene_id"):
        group_indices = group.index
        group_preds = preds[group_indices]
        pred_best = group_indices[np.argmax(group_preds)]
        true_best = group["quality_score1"].idxmax()
        correct += int(pred_best == true_best)
    acc = correct / df["scene_id"].nunique()
    selector_results_q1[name] = acc
    print(name,"acc:", acc) 

Ridge acc: 0.20717398592021455
Lasso acc: 0.20516258799865908
KNN acc: 0.21320817968488098
RF acc: 0.20683875293328863
GB acc: 0.1964465303385853


In [ ]:
# KNN is the highest accuracy model here so i won't bother with feature importances until Q2

### Training selector model regression on Q2 (action score)
* Repeating steps from Q1

In [ ]:
# reset to Q2
y = df["quality_score2"].values
all_preds = {}

In [79]:
# ridge
alphas = np.logspace(-3, 3, 50)
rcv = RidgeCV(alphas=alphas, cv=cv, scoring="neg_mean_absolute_error")
rcv.fit(X_scaled, y)
preds_ridge, _, _ = cv_metrics(rcv, X_scaled, y, cv, "Ridge")
all_preds["Ridge"] = preds_ridge

In [80]:
# lasso
lcv = LassoCV(alphas=alphas, cv=cv, max_iter=5000)
lcv.fit(X_scaled, y)
preds_lasso, _, _ = cv_metrics(lcv, X_scaled, y, cv, "Lasso")
all_preds["Lasso"] = preds_lasso

In [82]:
# KNN
Ks = np.arange(5, 51, 5)
knn_scores = []
for K in Ks:
    s = np.mean(cross_val_predict(
        KNeighborsRegressor(n_neighbors=K, weights="distance"),
        X_scaled, y, cv=cv
    ))
    knn_scores.append(s)
best_K = Ks[np.argmax(knn_scores)]
knn = KNeighborsRegressor(n_neighbors=best_K, weights="distance")
preds_knn, _, _ = cv_metrics(knn, X_scaled, y, cv, "KNN")
all_preds["KNN"] = preds_knn

In [83]:
# rf
rf_base = RandomForestRegressor(n_estimators=100, random_state=12)
rf_grid = {
    "max_samples": [0.6, 0.75, 0.9],
    "max_features": [0.3, 0.5, 0.7],
    "max_depth": [10, 20, None],
    "min_samples_leaf": [1, 3, 5]
}
rscv_rf = RandomizedSearchCV(rf_base, rf_grid, n_iter=5, cv=cv,
                              scoring="neg_mean_absolute_error",
                              random_state=12)
rscv_rf.fit(X, y)
preds_rf, _, _ = cv_metrics(rscv_rf.best_estimator_, X, y, cv, "RandomForest")
all_preds["RF"] = preds_rf

In [84]:
# xgb
gb_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.05, 0.1, 0.15],
    "max_depth": [3, 4, 5],
    "subsample": [0.75, 0.9],
    "min_samples_leaf": [1, 3],
    "max_features": [0.5, 0.7]
}
rscv_xgb = RandomizedSearchCV(
    XGBRegressor(random_state=12),
    xgb_grid, n_iter=5, cv=cv,
    scoring="neg_mean_absolute_error",
    random_state=12
)
rscv_xgb.fit(X, y)
preds_xgb, _, _ = cv_metrics(rscv_xgb.best_estimator_, X, y, cv, "XGB")
all_preds["GB"] = preds_xgb

In [85]:
# accuracies printed together
selector_results_q1 = {}

for name, preds in all_preds.items():
    correct = 0
    for _, group in df.groupby("scene_id"):
        group_indices = group.index
        group_preds = preds[group_indices]
        pred_best = group_indices[np.argmax(group_preds)]
        true_best = group["quality_score2"].idxmax()
        correct += int(pred_best == true_best)
    acc = correct / df["scene_id"].nunique()
    selector_results_q1[name] = acc
    print(name,"acc:", acc) 

Ridge acc: 0.2929936305732484
Lasso acc: 0.3003687562856185
KNN acc: 0.24472008045591687
RF acc: 0.2970164264163594
GB acc: 0.2929936305732484


### Phase C rendering

In [86]:
# picking the first 5 clips in my dataset
names = [
    "friends_s01e01_seg02_clip_04",
    "friends_s01e01_seg02_clip_19",
    "friends_s01e01_seg02_clip_27",
    "friends_s01e02_seg02_clip_00",
    "friends_s01e02_seg02_clip_05"
]

phaseA_filtered = phaseA[phaseA["tvad_name"].isin(names)].copy()

In [ ]:
# making a df of best candidates for those clips so i can edit the video manually
# i'd rather do a more automatic version of this but i'm out of time
df_best = (
    phaseA_filtered.assign(group=phaseA_filtered.index // 5)
      .loc[lambda x: x.groupby("group")["quality_score2"].idxmax()]
      .drop(columns="group")
      .reset_index(drop=True)
)

# convert AD_start from seconds to MM:SS timestamps
df_best["AD_start"] = pd.to_datetime(
    df_best["AD_start"], unit="s"
).dt.strftime("%M:%S")

df_best = df_best[["tvad_name", "AD_start", "human_AD_text", "gen_AD_text"]]

In [95]:
pd.set_option("display.max_colwidth", None)
df_best.head(20)

,tvad_name,AD_start,human_AD_text,gen_AD_text
0,friends_s01e01_seg02_clip_04,04:00,His umbrella springs open between them.,"Rachel kneels, holding a cup, as Phoebe gazes."
1,friends_s01e01_seg02_clip_04,04:02,"He awkwardly sits down, and Joey pats his shoulder.","Phoebe sits on the couch, surrounded by Monica, Ross, and Joey."
2,friends_s01e01_seg02_clip_04,04:05,Rachel sits next to him.,Rachel sits with her hands.
3,friends_s01e01_seg02_clip_19,15:03,Monica beams and sidles back inside.,"Monica, surrounded by friends, holds a cup."
4,friends_s01e01_seg02_clip_19,15:06,She shuts the door behind her.,Chandler irons while Monica talks.
5,friends_s01e01_seg02_clip_27,21:32,Rachel smiles and stands.,Rachel gets up from the couch.
6,friends_s01e01_seg02_clip_27,21:37,She heads into her bedroom.,Rachel walks towards the kitchen.
7,friends_s01e02_seg02_clip_00,02:44,Marsha slumps off as Ross waves Carol into the exhibit.,"Ross walks towards a caveman statue, then talks to a woman and man."
8,friends_s01e02_seg02_clip_00,02:48,"He tries to adjust the caveman, and the arm falls off.","Ross engages in a conversation, then leaves."
9,friends_s01e02_seg02_clip_00,02:51,He picks it up and uses it to wave at Carol.,"Ross converses with a woman in a blue dress, then hits the ground."
